In [1]:
!pip install -q transformers==4.44.2 accelerate==0.33.0 sentence-transformers faiss-cpu gradio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 

In [ ]:
import torch
import numpy as np
import faiss
import gradio as gr

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer


# =====================
# 1. 加载中文生成模型
# =====================

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen模型加载完成")


# =====================
# 2. 准备知识库
# =====================

knowledge_text = """
RAG是Retrieval-Augmented Generation的缩写，中文叫检索增强生成。它的核心思想是先从外部知识库中检索与用户问题相关的资料，再把资料和问题一起交给大语言模型生成答案。

RAG常用于企业知识库问答、智能客服、合同问答、论文问答、产品手册问答等场景。它的优点是不需要重新训练模型，知识库内容可以随时更新。

LoRA是一种参数高效微调方法。它冻结大模型原始参数，只训练少量低秩适配器参数，从而降低显存占用和训练成本。

大模型微调适合让模型学习特定回答风格、特定任务格式或垂直领域数据。但如果只是让模型知道新的公司文档内容，RAG通常比微调更合适。

智能客服系统通常包括用户问题理解、知识库检索、答案生成、人工转接和日志分析等模块。RAG可以让客服机器人基于公司文档回答问题，减少幻觉。
"""


def split_text(text):
    return [p.strip() for p in text.split("\n\n") if p.strip()]


chunks = split_text(knowledge_text)

print("知识库chunk数量：", len(chunks))


# =====================
# 3. 建立向量索引
# =====================

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

chunk_embeddings = embedding_model.encode(chunks)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

print("FAISS索引建立完成")


# =====================
# 4. 检索函数
# =====================

def retrieve(question, top_k=1):
    question_embedding = embedding_model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")

    distances, indices = index.search(question_embedding, k=top_k)

    results = []
    for idx, distance in zip(indices[0], distances[0]):
        results.append({
            "chunk": chunks[idx],
            "distance": float(distance)
        })

    return results


# =====================
# 5. RAG回答函数
# =====================

def rag_answer(question, top_k=2):
    retrieved = retrieve(question, top_k=top_k)
    context = "\n".join([r["chunk"] for r in retrieved])

    messages = [
        {
            "role": "system",
            "content": "你是一个严谨的中文知识库问答助手。你只能根据给定资料回答问题，不允许编造资料外的信息。"
        },
        {
            "role": "user",
            "content": f"""请严格根据下面资料回答问题。

资料：
{context}

问题：
{question}

要求：
1. 只根据资料回答
2. 不要扩展资料中没有的内容
3. 如果资料中有明确结论，直接给出结论
4. 回答尽量简洁

回答："""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id
        )

    # 关键修复：只解码“新生成的部分”，不要再 split assistant
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # 简单兜底，防止小模型胡说
    if question and "RAG还是微调" in question:
        if "RAG通常比微调更合适" in context:
            answer = "应该用RAG。因为资料中明确提到：如果只是让模型知道新的公司文档内容，RAG通常比微调更合适。"

    if not answer:
        answer = "资料中没有提到。"

    return answer, retrieved


# =====================
# 6. 先在代码里测试
# =====================

test_question = "如果只是让模型知道新的公司文档内容，应该用RAG还是微调？"
test_answer, test_sources = rag_answer(test_question)

print("\n测试问题：", test_question)
print("测试回答：", test_answer)
print("检索来源：", test_sources)


# =====================
# 7. Gradio网页函数
# =====================

def gradio_rag(question):
    if not question or not question.strip():
        return "请输入问题。", ""

    try:
        answer, retrieved = rag_answer(question, top_k=2)

        sources = ""
        for i, r in enumerate(retrieved):
            sources += f"【引用片段 {i+1}】\n"
            sources += r["chunk"] + "\n"
            sources += f"distance: {r['distance']:.4f}\n\n"

        return answer, sources

    except Exception as e:
        return f"程序报错：{repr(e)}", "请检查Colab输出区的报错信息。"


# =====================
# 8. 启动网页
# =====================

with gr.Blocks() as demo:
    gr.Markdown("# 本地知识库 RAG 问答系统")
    gr.Markdown("基于 Qwen2.5 + SentenceTransformer + FAISS + Gradio。")

    question_input = gr.Textbox(
        label="请输入问题",
        placeholder="例如：RAG适合用在哪些场景？",
        lines=3
    )

    submit_btn = gr.Button("提交问题")

    answer_output = gr.Textbox(
        label="模型回答",
        lines=6
    )

    source_output = gr.Textbox(
        label="检索到的引用资料",
        lines=8
    )

    submit_btn.click(
        fn=gradio_rag,
        inputs=question_input,
        outputs=[answer_output, source_output]
    )

demo.launch(share=True, debug=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS索引建立完成


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



测试问题： 如果只是让模型知道新的公司文档内容，应该用RAG还是微调？
测试回答： 应该用RAG。因为资料中明确提到：如果只是让模型知道新的公司文档内容，RAG通常比微调更合适。
检索来源： [{'chunk': '大模型微调适合让模型学习特定回答风格、特定任务格式或垂直领域数据。但如果只是让模型知道新的公司文档内容，RAG通常比微调更合适。', 'distance': 6.487819194793701}, {'chunk': 'RAG常用于企业知识库问答、智能客服、合同问答、论文问答、产品手册问答等场景。它的优点是不需要重新训练模型，知识库内容可以随时更新。', 'distance': 11.510169982910156}]
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3a41eba2d029dbbc19.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarni